In [1]:
from pathlib import Path
import pandas as pd
import numpy as np


DATA_DIR = Path.cwd() / "data"

CASUALTY_FILE = DATA_DIR / "dft-road-casualty-statistics-casualty-2025.csv"
COLLISON_FILE = DATA_DIR / "dft-road-casualty-statistics-collision-2025.csv"
VEHICLE_FILE = DATA_DIR / "dft-road-casualty-statistics-vehicle-2025.csv"

casualty_df = pd.read_csv(CASUALTY_FILE)
collision_df = pd.read_csv(COLLISON_FILE)
vehicle_df = pd.read_csv(VEHICLE_FILE)


C:\Users\HanifahAfinowi\AppData\Local\Temp\ipykernel_22104\2647172716.py:14: DtypeWarning: Columns (0: generic_make_model) have mixed types. Specify dtype option on import or set low_memory=False.
  vehicle_df = pd.read_csv(VEHICLE_FILE)


## Exploration

In [2]:
casualty_df.dtypes
casualty_df.shape # (127883, 24)

collision_df.dtypes
collision_df.shape # (101525, 44)

vehicle_df.dtypes
vehicle_df.shape # (183948, 32)

vehicle_unique = vehicle_df.groupby(["collision_index", "vehicle_reference"]).size()
casualty_unique = casualty_df.groupby(["collision_index", "casualty_reference"]).size()
print(casualty_unique)

collision_index  casualty_reference
2025010551784    1                     1
2025010551786    1                     1
2025010551792    1                     1
2025010551794    1                     1
                 2                     1
                                      ..
2025991732815    1                     1
2025991736589    1                     1
2025991742597    1                     1
                 2                     1
2025991744956    1                     1
Length: 127883, dtype: int64


In [ ]:
# verfying composite keys 
assert collision_df["collision_index"].is_unique

assert not vehicle_df.duplicated(
    ["collision_index", "vehicle_reference"]
).any()

assert not casualty_df.duplicated(
    ["collision_index", "casualty_reference"]
).any()

# Collision should be unique
assert collision_df["collision_index"].is_unique

# Each vehicle should be unique within a collision
assert not vehicle_df.duplicated(
    ["collision_index", "vehicle_reference"]
).any()

In [5]:
# Cheking relationship between casualty and vehicle records 
casualty_vehicle = casualty_df.merge(
    vehicle_df,
    on=["collision_index", "vehicle_reference"],
    how="left",
    validate="many_to_one",
    indicator=True
)

print(casualty_vehicle["_merge"].value_counts())

_merge
both          127883
left_only          0
right_only         0
Name: count, dtype: int64


In [6]:
casualty_summary = (
    casualty_df.groupby("collision_index")
    .agg(
        casualty_count=("casualty_reference", "size"),
        serious_casualties=(
            "casualty_severity",
            lambda x: (x == 2).sum()
        )
    )
    .reset_index()
)

collision_analysis = collision_df.merge(
    casualty_summary,
    on="collision_index",
    how="left",
    validate="one_to_one"
)

casualty_summary

,collision_index,casualty_count,serious_casualties
0,2025010551784,1,0
1,2025010551786,1,0
2,2025010551792,1,0
3,2025010551794,2,1
4,2025010551795,1,1
...,...,...,...
101520,2025991732325,1,0
101521,2025991732815,1,1
101522,2025991736589,1,0
101523,2025991742597,2,1


## Merging Datasets

In [7]:
# Join collisions to vehicles (one-to-many)
merged_df = collision_df.merge(
    vehicle_df,
    on="collision_index",
    how="left",
    validate="one_to_many",
    suffixes=("", "_vehicle")
)

# Join casualties to their associated vehicles
merged_df = merged_df.merge(
    casualty_df,
    on=["collision_index", "vehicle_reference"],
    how="left",
    validate="one_to_many",
    suffixes=("", "_casualty")
)

# Check the result
print("Merged shape:", merged_df.shape)
print("Unique collisions:", merged_df["collision_index"].nunique())
print("Unique vehicles:", merged_df[
    ["collision_index", "vehicle_reference"]
].drop_duplicates().shape[0])

Merged shape: (200247, 97)
Unique collisions: 101525
Unique vehicles: 183948


In [8]:
# Confirm no collisions were lost
assert (
    merged_df["collision_index"].nunique()
    == collision_df["collision_index"].nunique()
)

# Confirm all vehicles are represented
assert (
    merged_df[
        ["collision_index", "vehicle_reference"]
    ].drop_duplicates().shape[0]
    == len(vehicle_df)
)

# Confirm the number of matched casualty records
assert (
    merged_df["casualty_reference"].notna().sum()
    == len(casualty_df)
)

print("All merge validation checks passed.")

All merge validation checks passed.


Just thinking though, I don't know if it's worth using the orginal datasets because they have on row per area of interest (i.e. one row per collision, one orw per vehicle, oone row per casualty). With the merged dataset we then have one row per vehicle / casualty combination. So we could risk counting casualties multiple times 🤔